In [ ]:
from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path
import importlib.metadata
import json
import torch

PROJECT = Path("/content/drive/MyDrive/FacialVisualProfile")
ORACLE_ROOT = PROJECT / "runs/oracle_seed0"

print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

for name in ("torch", "torchvision", "numpy", "Pillow", "scikit-learn"):
    try:
        print(f"{name}: {importlib.metadata.version(name)}")
    except importlib.metadata.PackageNotFoundError:
        print(f"{name}: NOT INSTALLED")

paths = {
    "Oracle checkpoint": ORACLE_ROOT / "epoch_03.pt",
    "Oracle training config": ORACLE_ROOT / "config.json",
    "Oracle checkpoint selection": ORACLE_ROOT / "best.json",
    "Oracle model snapshot":
        ORACLE_ROOT / "source/models/resnet18_geometry.py",
    "Oracle saved thresholds":
        PROJECT / "runs/benchmark_preparation/day18/oracle_seed0.json",
    "Day19 report":
        PROJECT / "runs/benchmark/day19/day19_benchmark_report.json",
    "Day20 report":
        PROJECT / "runs/benchmark/day20/day20_benchmark_report.json",
}

for label, path in paths.items():
    print(f"{label}: exists={path.is_file()} | {path}")

for label in (
    "Oracle training config",
    "Oracle checkpoint selection",
    "Oracle saved thresholds",
):
    path = paths[label]
    if path.is_file():
        value = json.loads(path.read_bytes())
        print(f"\n{label} keys:", sorted(value))
        if label == "Oracle checkpoint selection":
            print(json.dumps(value, indent=2))
        elif label == "Oracle saved thresholds":
            selected = {
                key: value[key]
                for key in (
                    "checkpoint", "checkpoint_sha256",
                    "config_sha256", "fit_split", "samples",
                    "masked_validation_samples", "grid", "tie_break",
                    "comparison", "test_evaluated",
                )
                if key in value
            }
            print(json.dumps(selected, indent=2))

for day in ("day19", "day20"):
    path = PROJECT / f"runs/benchmark/{day}/{day}_benchmark_report.json"
    if path.is_file():
        report = json.loads(path.read_bytes())
        assert report[f"{day}_scope_complete"] is True
        assert report["threshold_fitting_performed"] is False
        print(f"{day}: completed conditions={report['condition_count']}")

print(
    "\nLocal test images available:",
    Path("/content/day19_benchmark_data/raw/img_align_celeba").is_dir(),
)
print("Day21 Oracle input inspection: COMPLETE")
print("No model inference or threshold fitting performed")

Mounted at /content/drive
CUDA available: False
torch: 2.11.0+cpu
torchvision: 0.26.0+cpu
numpy: 2.1.3
Pillow: 11.3.0
scikit-learn: 1.6.1
Oracle checkpoint: exists=True | /content/drive/MyDrive/FacialVisualProfile/runs/oracle_seed0/epoch_03.pt
Oracle training config: exists=True | /content/drive/MyDrive/FacialVisualProfile/runs/oracle_seed0/config.json
Oracle checkpoint selection: exists=True | /content/drive/MyDrive/FacialVisualProfile/runs/oracle_seed0/best.json
Oracle model snapshot: exists=True | /content/drive/MyDrive/FacialVisualProfile/runs/oracle_seed0/source/models/resnet18_geometry.py
Oracle saved thresholds: exists=True | /content/drive/MyDrive/FacialVisualProfile/runs/benchmark_preparation/day18/oracle_seed0.json
Day19 report: exists=True | /content/drive/MyDrive/FacialVisualProfile/runs/benchmark/day19/day19_benchmark_report.json
Day20 report: exists=True | /content/drive/MyDrive/FacialVisualProfile/runs/benchmark/day20/day20_benchmark_report.json

Oracle training config k

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path
import hashlib
import importlib.util
import json
import torch

PROJECT = Path("/content/drive/MyDrive/FacialVisualProfile")
ORACLE_ROOT = PROJECT / "runs/oracle_seed0"

assert torch.cuda.is_available(), "Select a GPU runtime"
print("GPU:", torch.cuda.get_device_name(0))


def sha_file(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as stream:
        for chunk in iter(lambda: stream.read(8 * 1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


threshold_path = (
    PROJECT / "runs/benchmark_preparation/day18/oracle_seed0.json"
)
assert sha_file(threshold_path) == (
    "71d7caed02170ac5a02ac0faf846748c"
    "465107ea2decb3faabdfe16922647ad3"
), "Oracle threshold file differs from the archived Day18 version"

ORACLE_THRESHOLDS = json.loads(threshold_path.read_bytes())
ORACLE_CONFIG = json.loads((ORACLE_ROOT / "config.json").read_bytes())

assert sha_file(ORACLE_ROOT / "config.json") == ORACLE_THRESHOLDS["config_sha256"]
assert ORACLE_THRESHOLDS["fit_split"] == "valid"
assert ORACLE_THRESHOLDS["samples"] == 19867
assert ORACLE_THRESHOLDS["masked_validation_samples"] == 0
assert ORACLE_THRESHOLDS["test_evaluated"] is False
assert ORACLE_THRESHOLDS["selected_epoch"] == 3
assert ORACLE_THRESHOLDS["comparison"] == ">="
assert ORACLE_THRESHOLDS["grid"] == [i / 100 for i in range(5, 96)]
assert ORACLE_THRESHOLDS["tie_break"] == "closest_to_0.5_then_lower"
assert len(ORACLE_THRESHOLDS["thresholds"]) == 24
assert all(
    value in ORACLE_THRESHOLDS["grid"]
    for value in ORACLE_THRESHOLDS["thresholds"]
)
assert (
    ORACLE_THRESHOLDS["attribute_names"]
    == ORACLE_CONFIG["attribute_names"]
)

for relative, expected in ORACLE_CONFIG["source_sha256"].items():
    assert sha_file(ORACLE_ROOT / "source" / relative) == expected, relative

checkpoint_path = ORACLE_ROOT / ORACLE_THRESHOLDS["checkpoint"]
assert ORACLE_THRESHOLDS["checkpoint_sha256"] == (
    "7552cb135a10d0ae72ee6faab7a5feb31"
    "c86ff9933165f96006f699656a06ab8"
)

print("Computing Oracle checkpoint SHA256...", flush=True)
assert sha_file(checkpoint_path) == ORACLE_THRESHOLDS["checkpoint_sha256"]

checkpoint = torch.load(
    checkpoint_path, map_location="cpu", weights_only=False
)
print("Checkpoint keys:", sorted(checkpoint))
assert checkpoint["epoch"] == 3
assert checkpoint["config"]["source_sha256"] == ORACLE_CONFIG["source_sha256"]
assert checkpoint["config"]["attribute_names"] == ORACLE_CONFIG["attribute_names"]

model_source = ORACLE_ROOT / "source/models/resnet18_geometry.py"
assert sha_file(model_source) == (
    "14786dc26945214a856495a367c7adab45eb4dd9ff486fd8a12720fec0fb501b"
), "Oracle model source checksum mismatch"

spec = importlib.util.spec_from_file_location("day21_oracle_model", model_source)
module = importlib.util.module_from_spec(spec)
spec.loader.exec_module(module)

ORACLE_MODEL = module.ResNet18GeometryAttributes(
    num_attributes=24, pretrained=False
)
state_keys = [
    key for key in ("model_state", "model_state_dict")
    if key in checkpoint
]
assert len(state_keys) == 1
ORACLE_MODEL.load_state_dict(checkpoint[state_keys[0]], strict=True)
ORACLE_MODEL = ORACLE_MODEL.to("cuda").eval()
ORACLE_STATE_KEY = state_keys[0]
del checkpoint

print("Geometry settings:")
print(json.dumps({
    key: ORACLE_CONFIG[key]
    for key in (
        "geometry_source", "geometry_valid", "failure_policy",
        "precision", "selection_metric", "selection_tie_rule",
    )
    if key in ORACLE_CONFIG
}, indent=2))

print("Oracle frozen thresholds, training snapshots and strict loading: PASS")
print("No model inference or threshold fitting performed")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
GPU: Tesla T4
Computing Oracle checkpoint SHA256...
Checkpoint keys: ['best', 'config', 'epoch', 'history', 'model_state_dict', 'optimizer_state_dict']
Geometry settings:
{
  "geometry_source": "CelebA aligned GT; shared preprocessing",
  "geometry_valid": "True for every train/validation sample",
  "failure_policy": "GT available; retain every sample",
  "precision": "FP32",
  "selection_metric": "validation_mAP",
  "selection_tie_rule": "earliest epoch"
}
Oracle frozen thresholds, training snapshots and strict loading: PASS
No model inference or threshold fitting performed


In [ ]:
from google.colab import files
import csv
import importlib.metadata
import io
import platform
import random
import zipfile

import numpy as np

uploaded = files.upload()
assert len(uploaded) == 1, "Upload only day21_oracle_frozen_inputs.zip"
archive_bytes = next(iter(uploaded.values()))
assert hashlib.sha256(archive_bytes).hexdigest() == (
    "122286cda1271ce2c62252a5b0735341c"
    "1a8cee0949cf35581f4e4bfc95fb946"
), "Oracle handoff archive checksum mismatch"


def preserve_file(path, data):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    if path.exists():
        assert path.read_bytes() == data, f"Existing file differs; preserved: {path}"
    else:
        temporary = path.with_suffix(path.suffix + ".writing")
        temporary.write_bytes(data)
        temporary.replace(path)


with zipfile.ZipFile(io.BytesIO(archive_bytes)) as archive:
    members = archive.namelist()
    assert len(members) == len(set(members))
    handoff = json.loads(archive.read("handoff_manifest.json"))
    assert set(members) == set(handoff["files"]) | {"handoff_manifest.json"}

    oracle_payloads = {}
    for name in members:
        relative = Path(name)
        assert not relative.is_absolute() and ".." not in relative.parts
        data = archive.read(name)
        if name != "handoff_manifest.json":
            assert hashlib.sha256(data).hexdigest() == handoff["files"][name], name
        oracle_payloads[name] = data

ORACLE_BINDING = json.loads(
    oracle_payloads["configs/oracle_benchmark_binding_v1.json"]
)
oracle_audit = json.loads(
    oracle_payloads[
        "docs/results/benchmark_freeze/day19/oracle_binding_audit.json"
    ]
)
additional = ORACLE_BINDING["additional_model"]

assert oracle_audit["passed"] is True
assert oracle_audit["day17_freeze_modified"] is False
assert additional["model_key"] == "oracle_seed0"
assert additional["detector_used"] is False
assert additional["checkpoint_sha256"] == ORACLE_THRESHOLDS["checkpoint_sha256"]
assert additional["thresholds"] == ORACLE_THRESHOLDS["thresholds"]
assert additional["source_sha256"] == ORACLE_CONFIG["source_sha256"]
assert additional["threshold_file_sha256"] == hashlib.sha256(
    oracle_payloads[additional["threshold_path"]]
).hexdigest()
assert (
    oracle_payloads[additional["threshold_path"]]
    == threshold_path.read_bytes()
)

DAY19 = PROJECT / "runs/benchmark/day19"
DAY20 = PROJECT / "runs/benchmark/day20"
DAY21 = PROJECT / "runs/benchmark/day21"
WORKSPACE = Path("/content/day21_benchmark_workspace")

D19_BINDING = json.loads((DAY19 / "execution_binding.json").read_bytes())

# Match the attribute-inference environment.
# facenet-pytorch is unnecessary because Oracle does not use a detector.
assert torch.cuda.is_available()
environment = {
    "python": platform.python_version(),
    "platform": platform.platform(),
    "gpu": torch.cuda.get_device_name(0),
    "cuda_runtime": torch.version.cuda,
    "cudnn_version": torch.backends.cudnn.version(),
}
for name, value in environment.items():
    assert value == D19_BINDING[name], (
        f"Environment differs: {name}; "
        f"Day19={D19_BINDING[name]!r}; current={value!r}"
    )
for name in ("torch", "torchvision", "numpy", "Pillow", "scikit-learn"):
    assert importlib.metadata.version(name) == D19_BINDING["environment"][name], name

random.seed(0)
np.random.seed(0)
torch.manual_seed(0)
torch.cuda.manual_seed_all(0)
torch.set_num_threads(D19_BINDING["torch_threads"])
torch.backends.cudnn.deterministic = D19_BINDING["cudnn_deterministic"]
torch.backends.cudnn.benchmark = D19_BINDING["cudnn_benchmark"]
torch.backends.cuda.matmul.allow_tf32 = D19_BINDING["tf32"]
torch.backends.cudnn.allow_tf32 = D19_BINDING["tf32"]

# Restore the exact Day17/Day19 frozen files.
saved_sources = DAY19 / "source/frozen_inputs"
base_handoff = json.loads(
    (saved_sources / "day19_handoff_manifest.json").read_bytes()
)
for name, expected in base_handoff["files"].items():
    relative = Path(name)
    assert not relative.is_absolute() and ".." not in relative.parts
    source = saved_sources / relative
    assert sha_file(source) == expected, name
    preserve_file(WORKSPACE / relative, source.read_bytes())

for path_key, hash_key in (
    ("base_config_path", "base_config_sha256"),
    ("base_freeze_audit_path", "base_freeze_audit_sha256"),
    ("test_manifest_path", "test_manifest_sha256"),
):
    assert sha_file(WORKSPACE / ORACLE_BINDING[path_key]) == ORACLE_BINDING[hash_key]

CONFIG = json.loads(
    (WORKSPACE / ORACLE_BINDING["base_config_path"]).read_bytes()
)
freeze = json.loads(
    (WORKSPACE / ORACLE_BINDING["base_freeze_audit_path"]).read_bytes()
)
assert freeze["passed"] is True
for name, expected in freeze["source_and_artifact_sha256"].items():
    assert sha_file(WORKSPACE / name) == expected, name
assert CONFIG["attribute_names"] == ORACLE_THRESHOLDS["attribute_names"]
assert len(CONFIG["conditions"]) == 21

with (WORKSPACE / ORACLE_BINDING["test_manifest_path"]).open(newline="") as stream:
    TEST_NAMES = [row["sample_id"] for row in csv.DictReader(stream)]
assert len(TEST_NAMES) == len(set(TEST_NAMES)) == 19962

BASE_REPORTS = {}
for day, directory, count in (
    ("day19", DAY19, 9),
    ("day20", DAY20, 12),
):
    result = json.loads((directory / f"{day}_benchmark_report.json").read_bytes())
    assert result[f"{day}_scope_complete"] is True
    assert result["condition_count"] == count
    assert result["test_sample_count"] == 19962
    assert result["threshold_fitting_performed"] is False
    assert result["runtime_errors"] == 0
    assert result["attribute_names"] == CONFIG["attribute_names"]
    assert sha_file(directory / "execution_binding.json") == (
        result["execution_binding_sha256"]
    )
    BASE_REPORTS[day] = result

d20_binding = json.loads((DAY20 / "execution_binding.json").read_bytes())
assert sha_file(DAY19 / "day19_benchmark_report.json") == (
    d20_binding["day19_report_sha256"]
)
assert sha_file(DAY19 / "execution_binding.json") == (
    d20_binding["day19_execution_binding_sha256"]
)

# Preserve the uploaded binding and its committed-file manifest.
for name, data in oracle_payloads.items():
    preserve_file(DAY21 / "source/oracle_frozen_inputs" / name, data)
    if name != "handoff_manifest.json":
        preserve_file(WORKSPACE / name, data)

print("Oracle additive binding and archived thresholds: PASS")
print("Shared inference environment and frozen benchmark rules: MATCH")
print("Official test IDs: 19962")
print("Existing three-model conditions: 21")
print("No model inference or threshold fitting performed")

Saving day21_oracle_frozen_inputs.zip to day21_oracle_frozen_inputs.zip
Oracle additive binding and archived thresholds: PASS
Shared inference environment and frozen benchmark rules: MATCH
Official test IDs: 19962
Existing three-model conditions: 21
No model inference or threshold fitting performed


In [ ]:
import importlib
import os
import sys
import types

from PIL import Image

DATA_ROOT = Path("/content/day21_benchmark_data/raw")
image_root = DATA_ROOT / "img_align_celeba"
image_root.mkdir(parents=True, exist_ok=True)

directories = (
    PROJECT / "data",
    PROJECT / "data/raw",
    PROJECT / "data/celeba",
    PROJECT,
)

# Restore the exact annotation files used by Day19.
for name, expected in D19_BINDING["annotation_sha256"].items():
    destination = DATA_ROOT / name
    if destination.exists():
        assert sha_file(destination) == expected, name
        continue

    matches = [
        directory / name
        for directory in directories
        if (directory / name).is_file()
        and sha_file(directory / name) == expected
    ]
    assert matches, f"Original annotation not found: {name}"
    preserve_file(destination, matches[0].read_bytes())

jpeg_manifest_path = DAY19 / "test_jpeg_sha256.json"
assert sha_file(jpeg_manifest_path) == (
    D19_BINDING["test_jpeg_manifest_sha256"]
)
JPEG_HASHES = json.loads(jpeg_manifest_path.read_bytes())
assert set(JPEG_HASHES) == set(TEST_NAMES)

missing = [
    name for name in TEST_NAMES
    if not (image_root / name).is_file()
]

if missing:
    local_zip = Path("/content/day21_img_align_celeba.zip")
    if local_zip.exists():
        assert sha_file(local_zip) == D19_BINDING["image_zip_sha256"]
    else:
        candidates = [
            directory / "img_align_celeba.zip"
            for directory in directories
            if (directory / "img_align_celeba.zip").is_file()
        ]
        assert len(candidates) == 1, f"Ambiguous image ZIP: {candidates}"

        print("Copying original image ZIP from Drive...", flush=True)
        temporary = local_zip.with_suffix(".copying")
        digest = hashlib.sha256()
        copied = 0
        next_progress = 256 * 1024 * 1024

        with candidates[0].open("rb") as source, temporary.open("wb") as target:
            for chunk in iter(lambda: source.read(8 * 1024 * 1024), b""):
                digest.update(chunk)
                target.write(chunk)
                copied += len(chunk)
                if copied >= next_progress:
                    print(f"Copied {copied / 1024**2:.0f} MiB", flush=True)
                    next_progress += 256 * 1024 * 1024

        assert digest.hexdigest() == D19_BINDING["image_zip_sha256"]
        temporary.replace(local_zip)

    wanted = set(missing)
    with zipfile.ZipFile(local_zip) as archive:
        members = {}
        for member in archive.infolist():
            name = member.filename.rsplit("/", 1)[-1]
            if not member.is_dir() and name in wanted:
                assert name not in members, name
                members[name] = member.filename

        assert set(members) == wanted
        for index, name in enumerate(missing, 1):
            data = archive.read(members[name])
            assert hashlib.sha256(data).hexdigest() == JPEG_HASHES[name], name
            preserve_file(image_root / name, data)
            if index % 2000 == 0 or index == len(missing):
                print(f"Restored {index}/{len(missing)} test images", flush=True)

for name in TEST_NAMES:
    assert sha_file(image_root / name) == JPEG_HASHES[name], name
assert {
    path.name for path in image_root.iterdir() if path.is_file()
} == set(TEST_NAMES)

# Import the frozen preprocessing and corruption implementations.
package_name = "day21_frozen_datasets"
package = types.ModuleType(package_name)
package.__path__ = [str(WORKSPACE / "datasets")]
sys.modules[package_name] = package

preprocessing = importlib.import_module(
    package_name + ".benchmark_preprocessing"
)
corruptions = importlib.import_module(package_name + ".corruptions")
original_preprocessing = importlib.import_module(
    package_name + ".preprocessing"
)

# Reuse Oracle's verified training Dataset implementation.
dataset_source = ORACLE_ROOT / "source/datasets/celeba.py"
assert sha_file(dataset_source) == additional["source_sha256"]["datasets/celeba.py"]
assert sha_file(dataset_source) == D19_BINDING["dataset_source_sha256"]

previous = {
    name: sys.modules.get(name)
    for name in ("datasets", "datasets.preprocessing")
}
try:
    sys.modules["datasets"] = package
    sys.modules["datasets.preprocessing"] = original_preprocessing
    spec = importlib.util.spec_from_file_location("day21_celeba", dataset_source)
    dataset_module = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(dataset_module)
finally:
    for name, before in previous.items():
        if before is None:
            sys.modules.pop(name, None)
        else:
            sys.modules[name] = before

previous_directory = Path.cwd()
try:
    os.chdir(WORKSPACE)
    TEST_DATASET = dataset_module.CelebAAttributes(
        split="test", root=DATA_ROOT
    )
finally:
    os.chdir(previous_directory)

assert TEST_DATASET.names == TEST_NAMES
assert TEST_DATASET.attribute_names == CONFIG["attribute_names"]

TARGETS = np.stack([
    TEST_DATASET.targets[name].numpy() for name in TEST_NAMES
]).astype(np.float32)
assert TARGETS.shape == (19962, 24)
assert np.isin(TARGETS, [0, 1]).all()

# Verify the full-test positive counts against existing reports.
for result in BASE_REPORTS.values():
    for item in result["conditions"]:
        for model_metrics in item["models"].values():
            np.testing.assert_array_equal(
                TARGETS.sum(axis=0),
                model_metrics["frozen_thresholds"]["positive_counts"],
            )

# Check clean preprocessing against the implementation used for training.
for name in TEST_NAMES[:4]:
    with Image.open(image_root / name) as image:
        old_image, old_points = original_preprocessing.prepare(
            image, TEST_DATASET.landmarks[name]
        )
        canvas, gt = preprocessing.prepare_canvas(
            image, TEST_DATASET.landmarks[name]
        )
    assert torch.equal(old_image, preprocessing.normalize_canvas(canvas))
    assert torch.equal(old_points, preprocessing.normalize_points(gt))

# Reconstruct four images under every condition and compare to saved records.
BASE_CONDITION_SOURCES = {}
for day, result in BASE_REPORTS.items():
    for item in result["conditions"]:
        condition = item["condition"]
        label = f"{condition['condition']}_s{condition['severity']}"
        assert label not in BASE_CONDITION_SOURCES
        BASE_CONDITION_SOURCES[label] = day

maximum_gt_error = 0.0
checked = 0

for condition in CONFIG["conditions"]:
    label = f"{condition['condition']}_s{condition['severity']}"
    day = BASE_CONDITION_SOURCES[label]
    directory = PROJECT / f"runs/benchmark/{day}"
    relative = f"shards/{label}/000000_000256.zip"
    shard_path = directory / relative

    assert sha_file(shard_path) == BASE_REPORTS[day]["shard_sha256"][relative]
    with zipfile.ZipFile(shard_path) as archive:
        manifest = json.loads(archive.read("manifest.json"))
        assert manifest["binding_sha256"] == (
            BASE_REPORTS[day]["execution_binding_sha256"]
        )
        assert manifest["condition"] == condition

        record_bytes = archive.read("records.json")
        prediction_bytes = archive.read("predictions.npz")
        assert hashlib.sha256(record_bytes).hexdigest() == manifest["files"]["records.json"]
        assert hashlib.sha256(prediction_bytes).hexdigest() == manifest["files"]["predictions.npz"]
        records = json.loads(record_bytes)

    with np.load(io.BytesIO(prediction_bytes), allow_pickle=False) as stored:
        assert stored["filenames"].tolist() == TEST_NAMES[:256]
        np.testing.assert_array_equal(stored["targets"], TARGETS[:256])

    for name, reference in zip(TEST_NAMES[:4], records[:4]):
        assert reference["sample_id"] == name
        with Image.open(image_root / name) as image:
            canvas, gt = preprocessing.prepare_canvas(
                image, TEST_DATASET.landmarks[name]
            )
        final_image, transformed_gt, metadata = corruptions.apply_corruption(
            canvas,
            condition["implementation_kind"],
            condition["severity"],
            sample_id=name,
            seed=CONFIG["corruption_seed"],
            gt_pixels=gt,
        )

        assert corruptions.rgb_sha256(final_image) == reference["final_rgb_sha256"]
        gt_array = np.asarray(transformed_gt, dtype=np.float64)
        np.testing.assert_allclose(
            gt_array, reference["gt_pixels"], rtol=0, atol=1e-12
        )
        maximum_gt_error = max(
            maximum_gt_error,
            float(np.max(np.abs(gt_array - reference["gt_pixels"]))),
        )
        points = preprocessing.normalize_points(transformed_gt)
        assert points.shape == (10,)
        assert torch.isfinite(points).all()
        checked += 1

print("Official test membership, attribute order and target counts: PASS")
print("Clean training/benchmark preprocessing equivalence: PASS")
print("Saved RGB and transformed-GT reproduction checks:", checked)
print("Maximum GT difference:", maximum_gt_error, "px")
print("Day21 Oracle data preparation: PASS")
print("No model inference or attribute metrics performed")

Copying original image ZIP from Drive...
Copied 256 MiB
Copied 512 MiB
Copied 768 MiB
Copied 1024 MiB
Copied 1280 MiB
Restored 2000/19962 test images
Restored 4000/19962 test images
Restored 6000/19962 test images
Restored 8000/19962 test images
Restored 10000/19962 test images
Restored 12000/19962 test images
Restored 14000/19962 test images
Restored 16000/19962 test images
Restored 18000/19962 test images
Restored 19962/19962 test images
Official test membership, attribute order and target counts: PASS
Clean training/benchmark preprocessing equivalence: PASS
Saved RGB and transformed-GT reproduction checks: 84
Maximum GT difference: 0.0 px
Day21 Oracle data preparation: PASS
No model inference or attribute metrics performed


In [ ]:
%%writefile /content/day21_evaluate_oracle.py
"""Evaluate frozen Oracle on 21 conditions using verified Day19/20 inputs."""
import hashlib
import importlib.metadata
import importlib.util
import io
import json
from pathlib import Path
import platform
import time
import traceback
import uuid
import zipfile

import numpy as np
from PIL import Image
import torch


def encoded(value):
    return (json.dumps(value, sort_keys=True, indent=2, allow_nan=False)
            + "\n").encode("utf-8")


def sha(data):
    return hashlib.sha256(data).hexdigest()


def sha_file(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as stream:
        for chunk in iter(lambda: stream.read(8 * 1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


def write_once(path, data):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    if path.exists():
        assert path.read_bytes() == data, f"Existing file differs; preserved: {path}"
        return
    temporary = path.with_name(path.name + "." + uuid.uuid4().hex + ".tmp")
    temporary.write_bytes(data)
    assert not path.exists(), f"Concurrent writer detected: {path}"
    temporary.replace(path)


def save_prediction(path, arrays, binding_sha, condition, reference_sha):
    buffer = io.BytesIO()
    np.savez(buffer, **arrays)
    data = buffer.getvalue()
    manifest = {
        "binding_sha256": binding_sha,
        "condition": condition,
        "reference_shard_sha256": reference_sha,
        "files": {"predictions.npz": sha(data)},
    }
    output = io.BytesIO()
    with zipfile.ZipFile(output, "w", zipfile.ZIP_DEFLATED) as archive:
        archive.writestr("predictions.npz", data)
        archive.writestr("manifest.json", encoded(manifest))
    write_once(path, output.getvalue())


def load_prediction(path, binding_sha, condition, reference_sha, names, targets,
                    reference_records, preprocessing):
    with zipfile.ZipFile(path) as archive:
        assert len(archive.namelist()) == 2
        assert set(archive.namelist()) == {"predictions.npz", "manifest.json"}
        manifest = json.loads(archive.read("manifest.json"))
        assert manifest["binding_sha256"] == binding_sha
        assert manifest["condition"] == condition
        assert manifest["reference_shard_sha256"] == reference_sha
        assert set(manifest["files"]) == {"predictions.npz"}
        data = archive.read("predictions.npz")
        assert sha(data) == manifest["files"]["predictions.npz"]
    with np.load(io.BytesIO(data), allow_pickle=False) as stored:
        arrays = {name: stored[name] for name in stored.files}

    assert arrays["filenames"].tolist() == names
    np.testing.assert_array_equal(arrays["targets"], targets)
    assert arrays["oracle_logits"].shape == (len(names), 24)
    assert arrays["oracle_logits"].dtype == np.float32
    assert np.isfinite(arrays["oracle_logits"]).all()
    assert arrays["geometry_valid"].dtype == np.bool_
    assert arrays["geometry_valid"].shape == (len(names),)
    assert arrays["geometry_valid"].all()
    assert arrays["gt_pixels"].shape == (len(names), 5, 2)
    assert arrays["gt_normalized_points"].shape == (len(names), 10)
    assert np.isfinite(arrays["gt_pixels"]).all()
    assert np.isfinite(arrays["gt_normalized_points"]).all()

    assert arrays["final_rgb_sha256"].tolist() == [
        row["final_rgb_sha256"] for row in reference_records
    ]
    np.testing.assert_allclose(
        arrays["gt_pixels"],
        [row["gt_pixels"] for row in reference_records],
        rtol=0, atol=1e-12,
    )
    for index in range(len(names)):
        np.testing.assert_array_equal(
            arrays["gt_normalized_points"][index],
            preprocessing.normalize_points(arrays["gt_pixels"][index]).numpy(),
        )
    return arrays


def run_oracle(*, workspace, project, dataset, model, preprocessing, corruptions):
    workspace, project = Path(workspace), Path(project)
    output = project / "runs/benchmark/day21"
    write_once(output / "source/scripts/day21_evaluate_oracle.py",
               Path(__file__).read_bytes())

    config_path = workspace / "configs/benchmark_v1.json"
    config = json.loads(config_path.read_bytes())
    binding_path = workspace / "configs/oracle_benchmark_binding_v1.json"
    oracle_binding = json.loads(binding_path.read_bytes())
    additional = oracle_binding["additional_model"]

    for path_key, hash_key in (
        ("base_config_path", "base_config_sha256"),
        ("base_freeze_audit_path", "base_freeze_audit_sha256"),
        ("test_manifest_path", "test_manifest_sha256"),
    ):
        assert sha_file(workspace / oracle_binding[path_key]) == oracle_binding[hash_key]

    freeze = json.loads(
        (workspace / oracle_binding["base_freeze_audit_path"]).read_bytes()
    )
    assert freeze["passed"]
    for name, expected in freeze["source_and_artifact_sha256"].items():
        assert sha_file(workspace / name) == expected, name

    threshold_path = workspace / additional["threshold_path"]
    assert sha_file(threshold_path) == additional["threshold_file_sha256"]
    thresholds = json.loads(threshold_path.read_bytes())
    assert thresholds["thresholds"] == additional["thresholds"]
    assert thresholds["attribute_names"] == config["attribute_names"]
    assert thresholds["fit_split"] == "valid"
    assert thresholds["test_evaluated"] is False

    oracle_root = project / "runs/oracle_seed0"
    training_path = oracle_root / "config.json"
    training = json.loads(training_path.read_bytes())
    assert sha_file(training_path) == thresholds["config_sha256"]
    assert training["source_sha256"] == additional["source_sha256"]
    for name, expected in additional["source_sha256"].items():
        assert sha_file(oracle_root / "source" / name) == expected, name
    assert sha_file(Path(additional["checkpoint_path"])) == additional["checkpoint_sha256"]
    assert thresholds["checkpoint_sha256"] == additional["checkpoint_sha256"]

    base_reports, base_bindings, references = {}, {}, {}
    base_report_hashes, base_binding_hashes = {}, {}
    for day, count in (("day19", 9), ("day20", 12)):
        directory = project / "runs/benchmark" / day
        report_path = directory / (day + "_benchmark_report.json")
        execution_path = directory / "execution_binding.json"
        result = json.loads(report_path.read_bytes())
        execution = json.loads(execution_path.read_bytes())
        assert result[day + "_scope_complete"] is True
        assert result["condition_count"] == count
        assert result["test_sample_count"] == 19962
        assert result["runtime_errors"] == 0
        assert result["threshold_fitting_performed"] is False
        assert result["attribute_names"] == config["attribute_names"]
        assert sha_file(execution_path) == result["execution_binding_sha256"]
        base_reports[day], base_bindings[day] = result, execution
        base_report_hashes[day] = sha_file(report_path)
        base_binding_hashes[day] = sha_file(execution_path)
        for item in result["conditions"]:
            condition = item["condition"]
            label = f"{condition['condition']}_s{condition['severity']}"
            assert label not in references
            references[label] = (day, condition)

    original = base_bindings["day19"]
    assert base_bindings["day20"]["day19_report_sha256"] == base_report_hashes["day19"]
    assert base_bindings["day20"]["day19_execution_binding_sha256"] == base_binding_hashes["day19"]
    for key in original:
        if key not in ("runner_sha256", "conditions"):
            assert original[key] == base_bindings["day20"][key], key

    expected_conditions = (
        [("clean", 0)]
        + [(kind, severity) for kind in
           ("blur", "brightness", "rotation", "occlusion", "jpeg")
           for severity in range(1, 5)]
    )
    conditions = config["conditions"]
    assert [(item["condition"], item["severity"]) for item in conditions] == expected_conditions
    for condition in conditions:
        label = f"{condition['condition']}_s{condition['severity']}"
        assert references[label][1] == condition

    assert len(dataset.names) == len(set(dataset.names)) == 19962
    assert dataset.attribute_names == config["attribute_names"]
    import csv
    with (workspace / oracle_binding["test_manifest_path"]).open(newline="") as stream:
        names = [row["sample_id"] for row in csv.DictReader(stream)]
    assert names == dataset.names
    targets = np.stack([dataset.targets[name].numpy() for name in names]).astype(np.float32)

    for name, expected in original["annotation_sha256"].items():
        assert sha_file(dataset.root / name) == expected, name
    jpeg_path = project / "runs/benchmark/day19/test_jpeg_sha256.json"
    assert sha_file(jpeg_path) == original["test_jpeg_manifest_sha256"]
    jpeg_hashes = json.loads(jpeg_path.read_bytes())
    assert set(jpeg_hashes) == set(names)
    write_once(output / "test_jpeg_sha256.json", jpeg_path.read_bytes())

    assert torch.cuda.is_available() and not model.training
    assert not torch.is_autocast_enabled()
    assert torch.get_num_threads() == original["torch_threads"]
    assert torch.backends.cudnn.deterministic == original["cudnn_deterministic"]
    assert torch.backends.cudnn.benchmark == original["cudnn_benchmark"]
    assert torch.backends.cuda.matmul.allow_tf32 == original["tf32"]
    assert torch.backends.cudnn.allow_tf32 == original["tf32"]
    assert all(p.device.type == "cuda" and p.dtype == torch.float32
               for p in model.parameters())

    environment = {
        "python": platform.python_version(), "platform": platform.platform(),
        "gpu": torch.cuda.get_device_name(0), "cuda_runtime": torch.version.cuda,
        "cudnn_version": torch.backends.cudnn.version(),
    }
    for key, value in environment.items():
        assert value == original[key], key
    packages = {name: importlib.metadata.version(name) for name in
                ("torch", "torchvision", "numpy", "Pillow", "scikit-learn")}
    for name, version in packages.items():
        assert version == original["environment"][name], name

    execution = {
        "runner_sha256": sha_file(__file__),
        "oracle_binding_sha256": sha_file(binding_path),
        "checkpoint_sha256": additional["checkpoint_sha256"],
        "threshold_file_sha256": sha_file(threshold_path),
        "training_config_sha256": sha_file(training_path),
        "source_sha256": additional["source_sha256"],
        "config_sha256": sha_file(config_path),
        "base_report_sha256": base_report_hashes,
        "base_execution_binding_sha256": base_binding_hashes,
        "test_manifest_sha256": oracle_binding["test_manifest_sha256"],
        "test_jpeg_manifest_sha256": sha_file(jpeg_path),
        "annotation_sha256": original["annotation_sha256"],
        "image_zip_sha256": original["image_zip_sha256"],
        "conditions": conditions, "attribute_names": config["attribute_names"],
        "environment": packages, "runtime": environment,
        "precision": "FP32", "batch_size": 16, "shard_size": 256,
        "torch_threads": original["torch_threads"],
        "cudnn_deterministic": True, "cudnn_benchmark": False, "tf32": False,
        "geometry_source": "transformed GT; all samples valid",
        "detector_used": False, "threshold_fitting_performed": False,
    }
    execution_bytes = encoded(execution)
    write_once(output / "execution_binding.json", execution_bytes)
    execution_sha = sha(execution_bytes)

    metric_path = workspace / "evaluation/benchmark_attributes.py"
    spec = importlib.util.spec_from_file_location("day21_frozen_metrics", metric_path)
    metrics = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(metrics)

    summaries, shard_hashes, reference_hashes = [], {}, {}
    active = {"stage": "start"}
    started = time.perf_counter()
    print("Oracle inference starts: 21 conditions, 19962 images; no detector", flush=True)

    try:
        for condition in conditions:
            label = f"{condition['condition']}_s{condition['severity']}"
            day = references[label][0]
            logits_parts = []
            print(f"\n=== {label} ===", flush=True)

            for start in range(0, len(names), 256):
                stop = min(start + 256, len(names))
                selected = names[start:stop]
                relative = f"shards/{label}/{start:06d}_{stop:06d}.zip"
                reference_path = project / "runs/benchmark" / day / relative
                reference_sha = base_reports[day]["shard_sha256"][relative]
                active = {"condition": condition, "start": start, "stop": stop,
                          "stage": "verify_reference"}
                assert sha_file(reference_path) == reference_sha, relative
                reference_hashes[day + "/" + relative] = reference_sha

                with zipfile.ZipFile(reference_path) as archive:
                    assert len(archive.namelist()) == 3
                    assert set(archive.namelist()) == {
                        "manifest.json", "records.json", "predictions.npz"
                    }
                    manifest = json.loads(archive.read("manifest.json"))
                    assert manifest["binding_sha256"] == base_binding_hashes[day]
                    assert manifest["condition"] == condition
                    assert set(manifest["files"]) == {"records.json", "predictions.npz"}
                    records_bytes = archive.read("records.json")
                    prediction_bytes = archive.read("predictions.npz")
                    assert sha(records_bytes) == manifest["files"]["records.json"]
                    assert sha(prediction_bytes) == manifest["files"]["predictions.npz"]
                    records = json.loads(records_bytes)

                with np.load(io.BytesIO(prediction_bytes), allow_pickle=False) as stored:
                    assert stored["filenames"].tolist() == selected
                    np.testing.assert_array_equal(stored["targets"], targets[start:stop])
                assert len(records) == len(selected)
                for name, row in zip(selected, records):
                    assert row["sample_id"] == name
                    assert row["condition"] == condition["condition"]
                    assert row["severity"] == condition["severity"]

                path = output / relative
                reused = path.exists()
                if not reused:
                    images, points, gt_values, rgb_hashes = [], [], [], []
                    for name, reference in zip(selected, records):
                        active.update(stage="reconstruct_input", sample_id=name)
                        raw_bytes = (dataset.root / "img_align_celeba" / name).read_bytes()
                        assert sha(raw_bytes) == jpeg_hashes[name], name
                        with Image.open(io.BytesIO(raw_bytes)) as image:
                            canvas, gt = preprocessing.prepare_canvas(
                                image, dataset.landmarks[name]
                            )
                        image, gt, metadata = corruptions.apply_corruption(
                            canvas, condition["implementation_kind"],
                            condition["severity"], sample_id=name,
                            seed=config["corruption_seed"], gt_pixels=gt,
                        )
                        rgb_sha = corruptions.rgb_sha256(image)
                        assert rgb_sha == reference["final_rgb_sha256"], name
                        np.testing.assert_allclose(
                            gt, reference["gt_pixels"], rtol=0, atol=1e-12
                        )
                        images.append(preprocessing.normalize_canvas(image))
                        points.append(preprocessing.normalize_points(gt))
                        gt_values.append(np.asarray(gt, dtype=np.float64))
                        rgb_hashes.append(rgb_sha)

                    images, points = torch.stack(images), torch.stack(points)
                    assert torch.isfinite(images).all() and torch.isfinite(points).all()
                    logits = []
                    active.update(stage="oracle_inference")
                    with torch.inference_mode():
                        for offset in range(0, len(selected), 16):
                            end = min(offset + 16, len(selected))
                            valid = torch.ones(end - offset, dtype=torch.bool, device="cuda")
                            values = model(
                                images[offset:end].to("cuda"),
                                points[offset:end].to("cuda"), valid,
                            )
                            assert values.shape == (end - offset, 24)
                            assert torch.isfinite(values).all()
                            logits.append(values.cpu().numpy())

                    arrays = {
                        "filenames": np.asarray(selected),
                        "targets": targets[start:stop],
                        "oracle_logits": np.concatenate(logits),
                        "geometry_valid": np.ones(len(selected), dtype=np.bool_),
                        "gt_normalized_points": points.numpy(),
                        "gt_pixels": np.stack(gt_values),
                        "final_rgb_sha256": np.asarray(rgb_hashes),
                    }
                    save_prediction(path, arrays, execution_sha, condition, reference_sha)

                arrays = load_prediction(
                    path, execution_sha, condition, reference_sha,
                    selected, targets[start:stop], records, preprocessing,
                )
                shard_hashes[relative] = sha_file(path)
                logits_parts.append(arrays["oracle_logits"])
                print(
                    f"{label}: {stop}/19962; {'reused' if reused else 'saved'}; "
                    f"elapsed={(time.perf_counter()-started)/60:.1f} min",
                    flush=True,
                )

            values = np.concatenate(logits_parts)
            frozen = metrics.evaluate_benchmark_attributes(
                values, targets, additional["thresholds"]
            )
            fixed = metrics.evaluate_benchmark_attributes(values, targets, [0.5] * 24)
            summary = {
                "condition": condition, "sample_count": len(names),
                "geometry_valid_count": len(names), "detector_used": False,
                "models": {"oracle_seed0": {
                    "frozen_thresholds": frozen, "fixed_0_5": fixed,
                }},
            }
            write_once(output / "condition_reports" / (label + ".json"), encoded(summary))
            summaries.append(summary)
            print(
                f"Oracle: F1(frozen)={frozen['macro_f1']:.6f}; "
                f"F1@0.5={fixed['macro_f1']:.6f}; mAP={frozen['map']:.6f}",
                flush=True,
            )

        for summary in summaries:
            values = summary["models"]["oracle_seed0"]
            for rule in ("frozen_thresholds", "fixed_0_5"):
                clean = summaries[0]["models"]["oracle_seed0"][rule]
                values[rule + "_degradation_pp"] = {
                    metric: 100 * (clean[metric] - values[rule][metric])
                    for metric in ("macro_f1", "map")
                }

        errors = {
            path.relative_to(output).as_posix(): sha_file(path)
            for path in sorted((output / "runtime_errors").glob("*.json"))
        }
        report = {
            "scope": "Oracle on all 21 frozen conditions",
            "execution_binding_sha256": execution_sha,
            "attribute_names": config["attribute_names"],
            "test_sample_count": len(names), "condition_count": 21,
            "model_sample_evaluations": len(names) * 21,
            "test_evaluated": True, "threshold_fitting_performed": False,
            "detector_used": False, "runtime_errors": 0,
            "prior_runtime_error_attempts": errors,
            "prior_error_resolution": "All required shards verified before completion",
            "conditions": summaries, "shard_sha256": shard_hashes,
            "reference_shard_sha256": reference_hashes,
            "oracle_scope_complete": True,
            "comparison_analysis_complete": False,
            "degradation_unit": "percentage points",
        }
        assert len(shard_hashes) == len(reference_hashes) == 1638
        write_once(output / "day21_oracle_report.json", encoded(report))
        print("Day21 Oracle formal evaluation: PASS", flush=True)
        return report

    except Exception:
        error = {
            "execution_binding_sha256": execution_sha,
            "active": active, "status": "runtime_error",
            "traceback": traceback.format_exc(), "completion_blocked": True,
        }
        write_once(
            output / "runtime_errors" / (uuid.uuid4().hex + ".json"),
            encoded(error),
        )
        print("Runtime error recorded; completed shards preserved.", flush=True)
        raise

Writing /content/day21_evaluate_oracle.py


In [6]:
runner_path = Path("/content/day21_evaluate_oracle.py")
spec = importlib.util.spec_from_file_location("day21_runner", runner_path)
oracle_runner = importlib.util.module_from_spec(spec)
spec.loader.exec_module(oracle_runner)

ORACLE_REPORT = oracle_runner.run_oracle(
    workspace=WORKSPACE,
    project=PROJECT,
    dataset=TEST_DATASET,
    model=ORACLE_MODEL,
    preprocessing=preprocessing,
    corruptions=corruptions,
)

Oracle inference starts: 21 conditions, 19962 images; no detector

=== clean_s0 ===
clean_s0: 256/19962; saved; elapsed=0.1 min
clean_s0: 512/19962; saved; elapsed=0.1 min
clean_s0: 768/19962; saved; elapsed=0.2 min
clean_s0: 1024/19962; saved; elapsed=0.2 min
clean_s0: 1280/19962; saved; elapsed=0.2 min
clean_s0: 1536/19962; saved; elapsed=0.3 min
clean_s0: 1792/19962; saved; elapsed=0.4 min
clean_s0: 2048/19962; saved; elapsed=0.4 min
clean_s0: 2304/19962; saved; elapsed=0.4 min
clean_s0: 2560/19962; saved; elapsed=0.5 min
clean_s0: 2816/19962; saved; elapsed=0.6 min
clean_s0: 3072/19962; saved; elapsed=0.6 min
clean_s0: 3328/19962; saved; elapsed=0.7 min
clean_s0: 3584/19962; saved; elapsed=0.7 min
clean_s0: 3840/19962; saved; elapsed=0.7 min
clean_s0: 4096/19962; saved; elapsed=0.8 min
clean_s0: 4352/19962; saved; elapsed=0.8 min
clean_s0: 4608/19962; saved; elapsed=0.9 min
clean_s0: 4864/19962; saved; elapsed=0.9 min
clean_s0: 5120/19962; saved; elapsed=1.0 min
clean_s0: 5376/1996

In [8]:
print("Condition        F1(frozen)   F1@0.5      mAP")
for item in ORACLE_REPORT["conditions"]:
    condition = item["condition"]
    label = f"{condition['condition']}_s{condition['severity']}"
    metrics = item["models"]["oracle_seed0"]
    frozen = metrics["frozen_thresholds"]
    fixed = metrics["fixed_0_5"]

    print(
        f"{label:16s} "
        f"{frozen['macro_f1']:.6f}     "
        f"{fixed['macro_f1']:.6f}    "
        f"{frozen['map']:.6f}"
    )

Condition        F1(frozen)   F1@0.5      mAP
clean_s0         0.756575     0.741867    0.820393
blur_s1          0.722567     0.682507    0.804209
blur_s2          0.549728     0.491136    0.745412
blur_s3          0.319232     0.270044    0.662828
blur_s4          0.152609     0.109967    0.560684
brightness_s1    0.751289     0.734930    0.819772
brightness_s2    0.741593     0.717906    0.815567
brightness_s3    0.714642     0.673232    0.804201
brightness_s4    0.588834     0.518580    0.765148
rotation_s1      0.738937     0.713791    0.811494
rotation_s2      0.709353     0.671522    0.798829
rotation_s3      0.672869     0.623799    0.784831
rotation_s4      0.628785     0.572151    0.766718
occlusion_s1     0.710430     0.689647    0.801508
occlusion_s2     0.570005     0.555234    0.747017
occlusion_s3     0.509131     0.473969    0.687057
occlusion_s4     0.408627     0.367639    0.621048
jpeg_s1          0.752817     0.731328    0.817359
jpeg_s2          0.747945     0.7202

In [9]:
import io
import zipfile
import numpy as np

BASE_REPORTS = {}
BASE_BINDINGS = {}
CONDITION_SOURCES = {}

for day in ("day19", "day20"):
    directory = PROJECT / f"runs/benchmark/{day}"
    result = json.loads(
        (directory / f"{day}_benchmark_report.json").read_bytes()
    )
    assert sha_file(directory / f"{day}_benchmark_report.json") == (
        ORACLE_EXECUTION["base_report_sha256"][day]
    )
    BASE_REPORTS[day] = result
    BASE_BINDINGS[day] = result["execution_binding_sha256"]
    for item in result["conditions"]:
        condition = item["condition"]
        label = f"{condition['condition']}_s{condition['severity']}"
        assert label not in CONDITION_SOURCES
        CONDITION_SOURCES[label] = (day, item)


def read_verified_zip(path, expected_sha, expected_members):
    data = path.read_bytes()
    assert hashlib.sha256(data).hexdigest() == expected_sha, str(path)
    with zipfile.ZipFile(io.BytesIO(data)) as archive:
        names = archive.namelist()
        assert len(names) == len(set(names))
        assert set(names) == expected_members
        manifest = json.loads(archive.read("manifest.json"))
        assert set(manifest["files"]) == expected_members - {"manifest.json"}
        payload = {
            name: archive.read(name)
            for name in manifest["files"]
        }
        for name, content in payload.items():
            assert hashlib.sha256(content).hexdigest() == manifest["files"][name]
    return manifest, payload


def read_npz(data):
    with np.load(io.BytesIO(data), allow_pickle=False) as stored:
        return {name: stored[name] for name in stored.files}


verified = 0
audit_conditions = []
canonical_names = None
canonical_targets = None
maximum_gt_error = 0.0
maximum_nme_error = 0.0

for oracle_summary in ORACLE_REPORT["conditions"]:
    condition = oracle_summary["condition"]
    label = f"{condition['condition']}_s{condition['severity']}"
    day, base_summary = CONDITION_SOURCES[label]
    assert base_summary["condition"] == condition

    names_parts, target_parts, nme_values = [], [], []
    status_counts = {"ok": 0, "no_face": 0, "invalid_output": 0}

    for start in range(0, 19962, 256):
        stop = min(start + 256, 19962)
        relative = f"shards/{label}/{start:06d}_{stop:06d}.zip"
        reference_sha = BASE_REPORTS[day]["shard_sha256"][relative]

        assert ORACLE_REPORT["reference_shard_sha256"][day + "/" + relative] == (
            reference_sha
        )

        base_manifest, base_payload = read_verified_zip(
            PROJECT / f"runs/benchmark/{day}" / relative,
            reference_sha,
            {"manifest.json", "predictions.npz", "records.json"},
        )
        oracle_manifest, oracle_payload = read_verified_zip(
            DAY21 / relative,
            ORACLE_REPORT["shard_sha256"][relative],
            {"manifest.json", "predictions.npz"},
        )

        assert base_manifest["binding_sha256"] == BASE_BINDINGS[day]
        assert oracle_manifest["binding_sha256"] == (
            ORACLE_REPORT["execution_binding_sha256"]
        )
        assert base_manifest["condition"] == oracle_manifest["condition"] == condition
        assert oracle_manifest["reference_shard_sha256"] == reference_sha

        base = read_npz(base_payload["predictions.npz"])
        oracle = read_npz(oracle_payload["predictions.npz"])
        records = json.loads(base_payload["records.json"])

        count = stop - start
        assert len(records) == count
        assert base["filenames"].shape == oracle["filenames"].shape == (count,)
        np.testing.assert_array_equal(base["filenames"], oracle["filenames"])
        np.testing.assert_array_equal(base["targets"], oracle["targets"])
        assert base["targets"].shape == (count, 24)
        assert np.isin(base["targets"], [0, 1]).all()
        assert oracle["geometry_valid"].shape == (count,)
        assert oracle["geometry_valid"].dtype == np.bool_
        assert oracle["geometry_valid"].all()
        assert base["geometry_valid"].shape == (count,)
        assert base["geometry_valid"].dtype == np.bool_
        assert base["predicted_normalized_points"].shape == (count, 10)
        assert np.isfinite(base["predicted_normalized_points"]).all()
        assert oracle["gt_pixels"].shape == (count, 5, 2)
        assert oracle["gt_normalized_points"].shape == (count, 10)
        assert np.isfinite(oracle["gt_pixels"]).all()

        expected_points = (
            2 * oracle["gt_pixels"].astype(np.float32) / 223 - 1
        ).reshape(count, 10)
        np.testing.assert_array_equal(
            oracle["gt_normalized_points"], expected_points
        )

        for run in ("r18_seed0", "r50_seed0", "m3_seed0"):
            values = base[run + "_logits"]
            assert values.shape == (count, 24)
            assert values.dtype == np.float32
            assert np.isfinite(values).all()

        assert oracle["oracle_logits"].shape == (count, 24)
        assert oracle["oracle_logits"].dtype == np.float32
        assert np.isfinite(oracle["oracle_logits"]).all()

        for index, record in enumerate(records):
            assert record["sample_id"] == base["filenames"][index]
            assert record["condition"] == condition["condition"]
            assert record["severity"] == condition["severity"]
            assert record["final_rgb_sha256"] == oracle["final_rgb_sha256"][index]

            gt = np.asarray(record["gt_pixels"], dtype=np.float64)
            np.testing.assert_allclose(
                oracle["gt_pixels"][index], gt, rtol=0, atol=1e-12
            )
            maximum_gt_error = max(
                maximum_gt_error,
                float(np.max(np.abs(oracle["gt_pixels"][index] - gt))),
            )

            status = record["detection"]["status"]
            assert status in status_counts
            status_counts[status] += 1
            valid = status == "ok"
            assert bool(base["geometry_valid"][index]) == valid

            if valid:
                landmarks = np.asarray(
                    record["detection"]["landmarks"], dtype=np.float64
                )
                assert landmarks.shape == (5, 2)
                assert np.isfinite(landmarks).all()
                eye_distance = float(np.linalg.norm(gt[0] - gt[1]))
                assert eye_distance > 0
                np.testing.assert_allclose(
                    eye_distance, record["gt_eye_distance_px"],
                    rtol=0, atol=1e-12,
                )
                nme = float(
                    np.linalg.norm(landmarks - gt, axis=1).mean()
                    / eye_distance
                )
                assert np.isfinite(record["nme"])
                np.testing.assert_allclose(
                    nme, record["nme"], rtol=0, atol=1e-12
                )
                maximum_nme_error = max(
                    maximum_nme_error, abs(nme - record["nme"])
                )
                nme_values.append(nme)
            else:
                assert record["nme"] is None
                assert not base["predicted_normalized_points"][index].any()

        names_parts.append(base["filenames"])
        target_parts.append(base["targets"])
        verified += 1
        if verified % 100 == 0:
            print(f"Verified paired shards: {verified}/1638", flush=True)

    names = np.concatenate(names_parts)
    targets = np.concatenate(target_parts)
    assert len(names) == len(set(names.tolist())) == 19962

    if canonical_names is None:
        canonical_names, canonical_targets = names, targets
    else:
        np.testing.assert_array_equal(names, canonical_names)
        np.testing.assert_array_equal(targets, canonical_targets)

    detector = base_summary["detector"]
    for status, count in status_counts.items():
        assert detector[status] == count
    assert detector["nme_count"] == len(nme_values)
    np.testing.assert_allclose(
        np.mean(nme_values), detector["mean_nme"], rtol=0, atol=1e-12
    )
    np.testing.assert_allclose(
        np.median(nme_values), detector["median_nme"], rtol=0, atol=1e-12
    )

    for summary in (base_summary, oracle_summary):
        for values in summary["models"].values():
            for rule in ("frozen_thresholds", "fixed_0_5"):
                np.testing.assert_array_equal(
                    targets.sum(axis=0), values[rule]["positive_counts"]
                )

    audit_conditions.append({
        "condition": condition,
        "samples": 19962,
        "paired_shards_verified": 78,
        "detector_status_counts": status_counts,
        "nme_count": len(nme_values),
        "mean_nme": float(np.mean(nme_values)),
        "median_nme": float(np.median(nme_values)),
    })
    print(f"{label}: paired inputs and detector NME PASS", flush=True)

assert verified == 1638

# Verify official sample order using the archived frozen manifest.
manifest_path = (
    PROJECT / "runs/benchmark/day19/source/frozen_inputs"
    / "docs/results/benchmark_freeze/day17/test_samples.csv"
)
assert sha_file(manifest_path) == ORACLE_EXECUTION["test_manifest_sha256"]
import csv
with manifest_path.open(newline="") as stream:
    official_names = [row["sample_id"] for row in csv.DictReader(stream)]
assert canonical_names.tolist() == official_names

PAIRED_AUDIT = {
    "oracle_report_sha256": sha_file(DAY21 / "day21_oracle_report.json"),
    "base_report_sha256": ORACLE_EXECUTION["base_report_sha256"],
    "oracle_shards_verified": verified,
    "reference_shards_verified": verified,
    "sample_count_per_condition": 19962,
    "condition_count": 21,
    "maximum_gt_difference_px": maximum_gt_error,
    "maximum_nme_reconstruction_difference": maximum_nme_error,
    "conditions": audit_conditions,
    "passed": True,
    "new_model_inference_performed": False,
    "threshold_fitting_performed": False,
}

audit_path = DAY21 / "analysis/paired_input_audit.json"
audit_bytes = (
    json.dumps(PAIRED_AUDIT, indent=2, allow_nan=False) + "\n"
).encode("utf-8")
audit_path.parent.mkdir(parents=True, exist_ok=True)
if audit_path.exists():
    assert audit_path.read_bytes() == audit_bytes, "Existing audit differs; preserved"
else:
    audit_path.write_bytes(audit_bytes)

print("\nVerified paired shards: 1638/1638")
print("Maximum GT difference:", maximum_gt_error, "px")
print("Maximum NME reconstruction difference:", maximum_nme_error)
print("Day21 paired prediction and NME audit: PASS")

clean_s0: paired inputs and detector NME PASS
Verified paired shards: 100/1638
blur_s1: paired inputs and detector NME PASS
Verified paired shards: 200/1638
blur_s2: paired inputs and detector NME PASS
Verified paired shards: 300/1638
blur_s3: paired inputs and detector NME PASS
blur_s4: paired inputs and detector NME PASS
Verified paired shards: 400/1638
brightness_s1: paired inputs and detector NME PASS
Verified paired shards: 500/1638
brightness_s2: paired inputs and detector NME PASS
Verified paired shards: 600/1638
brightness_s3: paired inputs and detector NME PASS
Verified paired shards: 700/1638
brightness_s4: paired inputs and detector NME PASS
rotation_s1: paired inputs and detector NME PASS
Verified paired shards: 800/1638
rotation_s2: paired inputs and detector NME PASS
Verified paired shards: 900/1638
rotation_s3: paired inputs and detector NME PASS
Verified paired shards: 1000/1638
rotation_s4: paired inputs and detector NME PASS
occlusion_s1: paired inputs and detector NM

In [10]:
from pathlib import Path
import csv
import hashlib
import io
import json
import numpy as np

PROJECT = Path("/content/drive/MyDrive/FacialVisualProfile")
OUTPUT = PROJECT / "runs/benchmark/day21/analysis"

def sha_file(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()

def read_json(path):
    return json.loads(path.read_text(encoding="utf-8"))

def encode_json(value):
    return (
        json.dumps(value, indent=2, allow_nan=False) + "\n"
    ).encode("utf-8")

def encode_csv(rows):
    stream = io.StringIO(newline="")
    writer = csv.DictWriter(stream, fieldnames=list(rows[0]))
    writer.writeheader()
    writer.writerows(rows)
    return stream.getvalue().encode("utf-8")

report_paths = {
    "day19": PROJECT / "runs/benchmark/day19/day19_benchmark_report.json",
    "day20": PROJECT / "runs/benchmark/day20/day20_benchmark_report.json",
    "day21": PROJECT / "runs/benchmark/day21/day21_oracle_report.json",
}
reports = {name: read_json(path) for name, path in report_paths.items()}

audit_path = OUTPUT / "paired_input_audit.json"
audit = read_json(audit_path)
assert audit["passed"] is True
assert audit["oracle_shards_verified"] == 1638
assert audit["reference_shards_verified"] == 1638
assert audit["oracle_report_sha256"] == sha_file(report_paths["day21"])
for day in ("day19", "day20"):
    assert audit["base_report_sha256"][day] == sha_file(report_paths[day])

attributes = reports["day21"]["attribute_names"]
model_names = ("r18_seed0", "r50_seed0", "m3_seed0", "oracle_seed0")
rules = ("frozen_thresholds", "fixed_0_5")

base_conditions = {}
for day in ("day19", "day20"):
    assert reports[day]["attribute_names"] == attributes
    for item in reports[day]["conditions"]:
        condition = item["condition"]
        label = f"{condition['condition']}_s{condition['severity']}"
        assert label not in base_conditions
        base_conditions[label] = item

overall_rows = []
attribute_rows = []
comparison_rows = []

for oracle_item in reports["day21"]["conditions"]:
    condition = oracle_item["condition"]
    label = f"{condition['condition']}_s{condition['severity']}"
    base_item = base_conditions[label]
    assert base_item["condition"] == condition

    models = {
        name: base_item["models"][name]
        for name in model_names[:3]
    }
    models["oracle_seed0"] = oracle_item["models"]["oracle_seed0"]

    for name in model_names:
        frozen = models[name]["frozen_thresholds"]
        fixed = models[name]["fixed_0_5"]
        assert abs(frozen["map"] - fixed["map"]) <= 1e-12

        overall_rows.append({
            "condition": label,
            "model": name,
            "macro_f1_frozen": frozen["macro_f1"],
            "macro_f1_at_0_5": fixed["macro_f1"],
            "map": frozen["map"],
        })

        for rule in rules:
            metrics = models[name][rule]
            f1 = np.asarray(metrics["per_attribute_f1"], dtype=np.float64)
            ap = np.asarray(metrics["per_attribute_ap"], dtype=np.float64)
            assert f1.shape == ap.shape == (24,)
            assert np.isfinite(f1).all() and np.isfinite(ap).all()
            assert abs(f1.mean() - metrics["macro_f1"]) <= 1e-12
            assert abs(ap.mean() - metrics["map"]) <= 1e-12

            for index, attribute in enumerate(attributes):
                attribute_rows.append({
                    "condition": label,
                    "model": name,
                    "threshold_rule": rule,
                    "attribute": attribute,
                    "f1": float(f1[index]),
                    "ap": float(ap[index]),
                    "positive_count": int(metrics["positive_counts"][index]),
                })

    for rule in rules:
        for left, right in (
            ("m3_seed0", "r18_seed0"),
            ("oracle_seed0", "m3_seed0"),
            ("oracle_seed0", "r18_seed0"),
            ("r50_seed0", "r18_seed0"),
        ):
            left_metrics = models[left][rule]
            right_metrics = models[right][rule]
            assert left_metrics["positive_counts"] == right_metrics["positive_counts"]

            for index, attribute in enumerate(attributes):
                comparison_rows.append({
                    "condition": label,
                    "threshold_rule": rule,
                    "attribute": attribute,
                    "model": left,
                    "reference_model": right,
                    "f1_difference_pp": 100 * (
                        left_metrics["per_attribute_f1"][index]
                        - right_metrics["per_attribute_f1"][index]
                    ),
                    "ap_difference_pp": 100 * (
                        left_metrics["per_attribute_ap"][index]
                        - right_metrics["per_attribute_ap"][index]
                    ),
                })

assert len(base_conditions) == 21
assert len(overall_rows) == 84
assert len(attribute_rows) == 21 * 4 * 2 * 24
assert len(comparison_rows) == 21 * 2 * 4 * 24

summary = {
    "input_report_sha256": {
        name: sha_file(path) for name, path in report_paths.items()
    },
    "paired_input_audit_sha256": sha_file(audit_path),
    "condition_count": 21,
    "model_names": list(model_names),
    "attribute_names": attributes,
    "threshold_rules": list(rules),
    "difference_unit": "percentage points",
    "overall_metrics": overall_rows,
    "new_model_inference": False,
    "threshold_fitting_performed": False,
    "note": (
        "Frozen thresholds are model-specific clean-validation thresholds. "
        "AP is independent of the threshold rule. "
        "Oracle is an independently trained GT-reference model; "
        "its performance is not a guaranteed upper bound."
    ),
}

payloads = {
    OUTPUT / "four_model_comparison.json": encode_json(summary),
    OUTPUT / "overall_metrics.csv": encode_csv(overall_rows),
    OUTPUT / "per_attribute_metrics.csv": encode_csv(attribute_rows),
    OUTPUT / "per_attribute_differences.csv": encode_csv(comparison_rows),
}

# Check all destinations before writing any output.
for path, data in payloads.items():
    if path.exists():
        assert path.read_bytes() == data, f"Existing file differs; preserved: {path}"

OUTPUT.mkdir(parents=True, exist_ok=True)
for path, data in payloads.items():
    path.write_bytes(data)

condition_order = [
    f"{item['condition']['condition']}_s{item['condition']['severity']}"
    for item in reports["day21"]["conditions"]
]
lookup = {(row["condition"], row["model"]): row for row in overall_rows}

for field, title in (
    ("macro_f1_frozen", "F1(frozen)"),
    ("macro_f1_at_0_5", "F1@0.5"),
    ("map", "mAP"),
):
    print(f"\n{title}")
    print(f"{'Condition':16s} {'R18':>10s} {'R50':>10s} {'M3':>10s} {'Oracle':>10s}")
    for label in condition_order:
        values = [
            lookup[(label, name)][field]
            for name in model_names
        ]
        print(f"{label:16s} " + " ".join(f"{value:10.6f}" for value in values))

print("\nOverall metric rows:", len(overall_rows))
print("Per-attribute metric rows:", len(attribute_rows))
print("Per-attribute comparison rows:", len(comparison_rows))
print("Saved:", OUTPUT)
print("Day21 four-model report comparison: PASS")


F1(frozen)
Condition               R18        R50         M3     Oracle
clean_s0           0.757701   0.756015   0.757834   0.756575
blur_s1            0.718908   0.715656   0.708930   0.722567
blur_s2            0.527141   0.531833   0.520733   0.549728
blur_s3            0.308625   0.304025   0.279198   0.319232
blur_s4            0.159067   0.176152   0.122907   0.152609
brightness_s1      0.754149   0.753333   0.754846   0.751289
brightness_s2      0.745119   0.746262   0.747840   0.741593
brightness_s3      0.716440   0.729098   0.725231   0.714642
brightness_s4      0.577975   0.648401   0.600987   0.588834
rotation_s1        0.743728   0.737617   0.741183   0.738937
rotation_s2        0.718881   0.714504   0.721719   0.709353
rotation_s3        0.682277   0.676971   0.698202   0.672869
rotation_s4        0.641866   0.641634   0.665834   0.628785
occlusion_s1       0.723695   0.719730   0.709241   0.710430
occlusion_s2       0.584136   0.590099   0.561994   0.570005
occlusion_s3

In [11]:
import io
import zipfile
from sklearn.metrics import f1_score

OUTPUT = PROJECT / "runs/benchmark/day21/analysis"
oracle_report = reports["day21"]

def load_shard(path, expected_sha256, expected_members):
    data = path.read_bytes()
    assert hashlib.sha256(data).hexdigest() == expected_sha256, (
        f"Shard hash mismatch: {path}"
    )

    with zipfile.ZipFile(io.BytesIO(data)) as archive:
        members = archive.namelist()
        assert len(members) == len(set(members))
        assert set(members) == set(expected_members)

        manifest = json.loads(archive.read("manifest.json"))
        for name, expected in manifest["files"].items():
            assert hashlib.sha256(archive.read(name)).hexdigest() == expected

        with np.load(
            io.BytesIO(archive.read("predictions.npz")),
            allow_pickle=False,
        ) as saved:
            arrays = {name: saved[name].copy() for name in saved.files}

        records = (
            json.loads(archive.read("records.json"))
            if "records.json" in members else None
        )

    return arrays, records, manifest

def stable_sigmoid(logits):
    values = np.asarray(logits, dtype=np.float64)
    result = np.empty_like(values)
    positive = values >= 0
    result[positive] = 1 / (1 + np.exp(-values[positive]))
    exp_values = np.exp(values[~positive])
    result[~positive] = exp_values / (1 + exp_values)
    return result

group_names = (
    "nme_le_5_percent",
    "nme_5_to_10_percent",
    "nme_10_to_20_percent",
    "nme_gt_20_percent",
    "detection_failure",
)

group_rows = []
group_attribute_rows = []
verified_pairs = 0

for oracle_item in oracle_report["conditions"]:
    condition = oracle_item["condition"]
    label = f"{condition['condition']}_s{condition['severity']}"
    day = "day19" if label in {
        f"{item['condition']['condition']}_s{item['condition']['severity']}"
        for item in reports["day19"]["conditions"]
    } else "day20"

    base_item = base_conditions[label]
    model_metrics = {
        name: base_item["models"][name]
        for name in model_names[:3]
    }
    model_metrics["oracle_seed0"] = oracle_item["models"]["oracle_seed0"]

    target_parts = []
    validity_parts = []
    nme_parts = []
    logit_parts = {name: [] for name in model_names}

    for start in range(0, 19962, 256):
        stop = min(start + 256, 19962)
        relative = f"shards/{label}/{start:06d}_{stop:06d}.zip"
        reference_key = f"{day}/{relative}"

        base_sha = reports[day]["shard_sha256"][relative]
        assert oracle_report["reference_shard_sha256"][reference_key] == base_sha

        base, records, base_manifest = load_shard(
            PROJECT / f"runs/benchmark/{day}" / relative,
            base_sha,
            ("predictions.npz", "records.json", "manifest.json"),
        )
        oracle, _, oracle_manifest = load_shard(
            PROJECT / "runs/benchmark/day21" / relative,
            oracle_report["shard_sha256"][relative],
            ("predictions.npz", "manifest.json"),
        )

        assert base_manifest["condition"] == condition
        assert oracle_manifest["condition"] == condition
        assert oracle_manifest["reference_shard_sha256"] == base_sha
        assert np.array_equal(base["filenames"], oracle["filenames"])
        assert np.array_equal(base["targets"], oracle["targets"])
        assert len(records) == stop - start

        valid = base["geometry_valid"]
        nme = np.array([
            float(record["nme"]) if record["nme"] is not None else np.nan
            for record in records
        ])
        assert np.array_equal(np.isfinite(nme), valid)
        assert (nme[valid] >= 0).all()

        target_parts.append(base["targets"])
        validity_parts.append(valid)
        nme_parts.append(nme)

        for name in model_names[:3]:
            logit_parts[name].append(base[f"{name}_logits"])
        logit_parts["oracle_seed0"].append(oracle["oracle_logits"])

        verified_pairs += 1
        if verified_pairs % 200 == 0:
            print(f"Loaded paired shards: {verified_pairs}/1638")

    targets = np.concatenate(target_parts).astype(np.int64)
    valid = np.concatenate(validity_parts)
    nme = np.concatenate(nme_parts)

    masks = {
        "nme_le_5_percent": valid & (nme <= 0.05),
        "nme_5_to_10_percent": valid & (nme > 0.05) & (nme <= 0.10),
        "nme_10_to_20_percent": valid & (nme > 0.10) & (nme <= 0.20),
        "nme_gt_20_percent": valid & (nme > 0.20),
        "detection_failure": ~valid,
    }
    assert np.all(
        np.stack(list(masks.values())).sum(axis=0) == 1
    )

    for name in model_names:
        logits = np.concatenate(logit_parts[name])
        assert logits.shape == (19962, 24)
        assert np.isfinite(logits).all()
        probabilities = stable_sigmoid(logits)

        for rule in rules:
            thresholds = np.asarray(
                model_metrics[name][rule]["thresholds"],
                dtype=np.float64,
            )
            assert thresholds.shape == (24,)
            predictions = (probabilities >= thresholds).astype(np.int64)

            # Reproduce the recorded full-condition F1 before subgroup analysis.
            full_f1 = f1_score(
                targets, predictions, average=None, zero_division=0
            )
            np.testing.assert_allclose(
                full_f1,
                model_metrics[name][rule]["per_attribute_f1"],
                rtol=0,
                atol=1e-12,
            )

            for group in group_names:
                mask = masks[group]
                count = int(mask.sum())
                group_targets = targets[mask]
                positive_counts = group_targets.sum(axis=0).tolist()

                if count:
                    attribute_f1 = f1_score(
                        group_targets,
                        predictions[mask],
                        average=None,
                        zero_division=0,
                    )
                    macro_f1 = float(attribute_f1.mean())
                else:
                    attribute_f1 = [None] * 24
                    macro_f1 = None

                group_rows.append({
                    "condition": label,
                    "group": group,
                    "samples": count,
                    "model": name,
                    "threshold_rule": rule,
                    "macro_f1": macro_f1,
                    "attributes_without_positive_samples": (
                        int(sum(value == 0 for value in positive_counts))
                    ),
                })

                for index, attribute in enumerate(attributes):
                    group_attribute_rows.append({
                        "condition": label,
                        "group": group,
                        "samples": count,
                        "model": name,
                        "threshold_rule": rule,
                        "attribute": attribute,
                        "positive_count": int(positive_counts[index]),
                        "f1": (
                            float(attribute_f1[index]) if count else None
                        ),
                    })

    print(f"{label}: both F1 rules reproduced; NME groups complete")

assert verified_pairs == 1638

analysis = {
    "input_report_sha256": {
        name: sha_file(path) for name, path in report_paths.items()
    },
    "paired_input_audit_sha256": sha_file(audit_path),
    "analysis_type": "exploratory descriptive subgroup comparison",
    "group_boundaries_nme_ratio": [0.05, 0.10, 0.20],
    "group_names": list(group_names),
    "threshold_rules": list(rules),
    "f1_average": "macro over all 24 attributes",
    "zero_division": 0,
    "empty_group_result": None,
    "group_metrics": group_rows,
    "notes": [
        "All four models are evaluated on identical samples within each group.",
        "NME is defined only for valid detector output.",
        "Detection failures are retained and analyzed separately.",
        "Groups can differ in attribute prevalence and sample difficulty.",
        "Attributes without positive samples can affect subgroup macro F1.",
        "These associations do not establish a causal effect of landmark error.",
        "No threshold fitting or new model inference was performed.",
    ],
}

payloads = {
    OUTPUT / "nme_group_analysis.json": encode_json(analysis),
    OUTPUT / "nme_group_metrics.csv": encode_csv(group_rows),
    OUTPUT / "nme_group_attribute_metrics.csv": encode_csv(
        group_attribute_rows
    ),
}

for path, data in payloads.items():
    if path.exists():
        assert path.read_bytes() == data, f"Existing file differs; preserved: {path}"

for path, data in payloads.items():
    path.write_bytes(data)

print("\nVerified paired shards:", verified_pairs)
print("Group metric rows:", len(group_rows))
print("Group attribute rows:", len(group_attribute_rows))
print("Saved:", OUTPUT)
print("Day21 NME subgroup analysis: PASS")

clean_s0: both F1 rules reproduced; NME groups complete
blur_s1: both F1 rules reproduced; NME groups complete
Loaded paired shards: 200/1638
blur_s2: both F1 rules reproduced; NME groups complete
blur_s3: both F1 rules reproduced; NME groups complete
blur_s4: both F1 rules reproduced; NME groups complete
Loaded paired shards: 400/1638
brightness_s1: both F1 rules reproduced; NME groups complete
brightness_s2: both F1 rules reproduced; NME groups complete
Loaded paired shards: 600/1638
brightness_s3: both F1 rules reproduced; NME groups complete
brightness_s4: both F1 rules reproduced; NME groups complete
rotation_s1: both F1 rules reproduced; NME groups complete
Loaded paired shards: 800/1638
rotation_s2: both F1 rules reproduced; NME groups complete
rotation_s3: both F1 rules reproduced; NME groups complete
Loaded paired shards: 1000/1638
rotation_s4: both F1 rules reproduced; NME groups complete
occlusion_s1: both F1 rules reproduced; NME groups complete
occlusion_s2: both F1 rules 

In [12]:
analysis = read_json(OUTPUT / "nme_group_analysis.json")
rows = analysis["group_metrics"]

lookup = {
    (
        row["condition"],
        row["group"],
        row["model"],
        row["threshold_rule"],
    ): row
    for row in rows
}

selected_conditions = (
    "clean_s0",
    "rotation_s4",
    "occlusion_s4",
    "blur_s4",
)

for condition in selected_conditions:
    print(f"\n=== {condition} ===")
    print("Each model: F1(frozen) / F1@0.5")
    print(
        f"{'NME group':24s} {'Samples':>7s} "
        f"{'R18':>17s} {'R50':>17s} "
        f"{'M3':>17s} {'Oracle':>17s}"
    )

    for group in analysis["group_names"]:
        reference = lookup[
            (condition, group, "r18_seed0", "frozen_thresholds")
        ]
        count = reference["samples"]
        cells = []

        for model in model_names:
            frozen = lookup[
                (condition, group, model, "frozen_thresholds")
            ]
            fixed = lookup[
                (condition, group, model, "fixed_0_5")
            ]
            assert frozen["samples"] == fixed["samples"] == count

            cells.append(
                f"{frozen['macro_f1']:.6f}/{fixed['macro_f1']:.6f}"
                if count else "empty"
            )

        print(
            f"{group:24s} {count:7d} "
            + " ".join(f"{cell:>17s}" for cell in cells)
        )

        if count and reference["attributes_without_positive_samples"]:
            print(
                "  Attributes without positive samples:",
                reference["attributes_without_positive_samples"],
            )

print("\nNME subgroup display: COMPLETE")


=== clean_s0 ===
Each model: F1(frozen) / F1@0.5
NME group                Samples               R18               R50                M3            Oracle
nme_le_5_percent            8237 0.763627/0.730792 0.762038/0.730776 0.765444/0.744124 0.763188/0.750366
nme_5_to_10_percent        10184 0.756977/0.722832 0.755978/0.723106 0.756007/0.735452 0.755328/0.738479
nme_10_to_20_percent        1249 0.706281/0.653820 0.701731/0.655930 0.701452/0.673080 0.700230/0.680093
nme_gt_20_percent            271 0.644573/0.606243 0.630588/0.613609 0.662275/0.630710 0.654562/0.637171
detection_failure             21 0.257688/0.246090 0.340476/0.307143 0.255952/0.236041 0.234010/0.218204
  Attributes without positive samples: 7

=== rotation_s4 ===
Each model: F1(frozen) / F1@0.5
NME group                Samples               R18               R50                M3            Oracle
nme_le_5_percent            5976 0.640994/0.564816 0.643581/0.566294 0.664699/0.605969 0.626418/0.570545
nme_5_to_10_perc

In [13]:
from google.colab import files
import zipfile

DAY21 = PROJECT / "runs/benchmark/day21"
OUTPUT = DAY21 / "analysis"

oracle_path = DAY21 / "day21_oracle_report.json"
execution_path = DAY21 / "execution_binding.json"
audit_path = OUTPUT / "paired_input_audit.json"

oracle_report = read_json(oracle_path)
execution = read_json(execution_path)
paired_audit = read_json(audit_path)
comparison = read_json(OUTPUT / "four_model_comparison.json")
nme_analysis = read_json(OUTPUT / "nme_group_analysis.json")

assert oracle_report["oracle_scope_complete"] is True
assert oracle_report["condition_count"] == 21
assert oracle_report["execution_binding_sha256"] == sha_file(execution_path)
assert paired_audit["passed"] is True
assert paired_audit["oracle_shards_verified"] == 1638
assert paired_audit["reference_shards_verified"] == 1638
assert paired_audit["oracle_report_sha256"] == sha_file(oracle_path)

input_hashes = {
    name: sha_file(path) for name, path in report_paths.items()
}
assert comparison["input_report_sha256"] == input_hashes
assert nme_analysis["input_report_sha256"] == input_hashes
assert comparison["paired_input_audit_sha256"] == sha_file(audit_path)
assert nme_analysis["paired_input_audit_sha256"] == sha_file(audit_path)

runner_path = DAY21 / "source/scripts/day21_evaluate_oracle.py"
assert sha_file(runner_path) == execution["runner_sha256"]

analysis_names = (
    "paired_input_audit.json",
    "four_model_comparison.json",
    "overall_metrics.csv",
    "per_attribute_metrics.csv",
    "per_attribute_differences.csv",
    "nme_group_analysis.json",
    "nme_group_metrics.csv",
    "nme_group_attribute_metrics.csv",
)

selected = {oracle_path, execution_path, runner_path}
for name in analysis_names:
    path = OUTPUT / name
    assert path.is_file(), f"Missing analysis output: {path}"
    selected.add(path)

condition_paths = sorted((DAY21 / "condition_reports").glob("*.json"))
assert len(condition_paths) == 21
selected.update(condition_paths)

# Include the saved source and frozen configuration files.
source_root = DAY21 / "source"
allowed_suffixes = {".py", ".json", ".csv", ".yaml", ".yml", ".md"}
for path in source_root.rglob("*"):
    if path.is_file() and path.suffix in allowed_suffixes:
        selected.add(path)

# Include a local JPEG hash manifest if the runner saved one.
jpeg_manifest = DAY21 / "test_jpeg_sha256.json"
if jpeg_manifest.is_file():
    assert sha_file(jpeg_manifest) == execution["test_jpeg_manifest_sha256"]
    selected.add(jpeg_manifest)

payloads = {
    path.relative_to(DAY21).as_posix(): path.read_bytes()
    for path in sorted(selected)
}

manifest = {
    "archive_scope": "Day21 Oracle evaluation and four-model comparison",
    "files": {
        name: hashlib.sha256(data).hexdigest()
        for name, data in payloads.items()
    },
    "input_report_sha256": input_hashes,
    "oracle_prediction_shards_verified": 1638,
    "reference_prediction_shards_verified": 1638,
    "shard_verification_evidence": "analysis/paired_input_audit.json",
    "threshold_rules": ["frozen_thresholds", "fixed_0_5"],
    "comparison_analysis_complete": True,
    "new_model_inference_during_archiving": False,
    "threshold_fitting_performed": False,
    "prediction_shards_included": False,
    "prediction_shard_location": "runs/benchmark/day21/shards/",
    "analysis_code_location": (
        "day21_oracle_benchmark_and_comparison.ipynb"
    ),
}

manifest_bytes = encode_json(manifest)
manifest_path = DAY21 / "archive_manifest.json"
if manifest_path.exists():
    assert manifest_path.read_bytes() == manifest_bytes, (
        "Existing archive manifest differs; preserved"
    )
else:
    manifest_path.write_bytes(manifest_bytes)

payloads["archive_manifest.json"] = manifest_bytes
archive_path = Path("/content/day21_results_archive.zip")

with zipfile.ZipFile(
    archive_path, "w", compression=zipfile.ZIP_DEFLATED
) as archive:
    for name, data in sorted(payloads.items()):
        archive.writestr(name, data)

with zipfile.ZipFile(archive_path) as archive:
    assert len(archive.namelist()) == len(payloads)
    assert set(archive.namelist()) == set(payloads)
    for name, data in payloads.items():
        assert archive.read(name) == data, f"Archive content differs: {name}"

print("Archived files:", len(payloads))
print("Archive SHA256:", sha_file(archive_path))
print("Day21 results archive: PASS")
files.download(str(archive_path))

Archived files: 38
Archive SHA256: 726b2ef444877cc2ed4dba51a80b9c5e535de17cc18e10c8ca11b146e0b7985c
Day21 results archive: PASS


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>